# One-time Setup (`setup_nbk`)

Builds everything the query pipeline depends on, using the `optimized_*` family only.
Run top-to-bottom **once per environment** (or whenever source data changes).

Steps:
0. Download models (embedding + reranker) to the UC Volume
1. Preprocess: build `optimized_rag_chunks` from the PROD lake
2. Create the Vector Search index `optimized_rag_chunks_vs`
3. Build the BM25 inverted-index tables (`optimized_kw_*`)


## 0. Download models to UC Volume

- Embedding model: `DMIR01/DMRetriever-33M` (query + document embeddings)
- Reranker (cross-encoder): `cross-encoder/ms-marco-MiniLM-L-6-v2`

Paths here must match `MODEL_DIR` / `RERANKER_DIR` in `scripts/`.

In [ ]:
from huggingface_hub import snapshot_download
import os

MODELS = [
    ("DMIR01/DMRetriever-33M",
     "/Volumes/tdis_dev_data_catalog/tdir/tdir/models/DMRetriever-33M"),
    ("cross-encoder/ms-marco-MiniLM-L-6-v2",
     "/Volumes/tdis_dev_data_catalog/tdir/tdir/models/ms-marco-MiniLM-L-6-v2"),
]

for repo_id, local_dir in MODELS:
    os.makedirs(local_dir, exist_ok=True)
    snapshot_download(repo_id=repo_id, local_dir=local_dir, local_dir_use_symlinks=False)
    print("Downloaded:", repo_id, "->", local_dir)
    display(dbutils.fs.ls(local_dir))

## 1. Preprocess: build `optimized_rag_chunks`

Reads pre-chunked text + pre-computed DMRetriever-33M embeddings from the PROD bronze lake
and writes three DEV tables: `optimized_chunks_text`, `optimized_embeddings_dmretriever33m`,
and the core join `optimized_rag_chunks`.

In [ ]:
base = "abfss://tdis-data-bronze@tdisproddatalakehouse.dfs.core.windows.net/RAG_files/"
display(dbutils.fs.ls(base + "optimized_chunks/"))
display(dbutils.fs.ls(base + "optimized_embeddings/DMRetriever-33M/"))

In [ ]:
import json, io
import numpy as np
import pandas as pd
from pyspark.sql.functions import col, regexp_extract, count, countDistinct
from pyspark.sql.types import StructType, StructField, StringType, ArrayType, FloatType

# Chunk text
df_chunks = (
    spark.read.json(base + "optimized_chunks/*/chunks.jsonl")
    .select("chunk_id", "source_file", "chunk_index_in_file", "text")
)

# 1) Read all embeddings.npy bytes
df_npy = (spark.read.format("binaryFile")
          .load(base + "optimized_embeddings/DMRetriever-33M/*/embeddings.npy")
          .withColumn("source_dir", regexp_extract(col("path"), r"/DMRetriever-33M/([^/]+)/embeddings\.npy$", 1))
          .select("source_dir", col("content").alias("npy_bytes")))

# 2) Read all chunk_ids.json bytes
df_ids = (spark.read.format("binaryFile")
          .load(base + "optimized_embeddings/DMRetriever-33M/*/chunk_ids.json")
          .withColumn("source_dir", regexp_extract(col("path"), r"/DMRetriever-33M/([^/]+)/chunk_ids\.json$", 1))
          .select("source_dir", col("content").alias("ids_bytes")))

# 3) Pair npy + ids per source dir
df_pair = (df_npy.join(df_ids, on="source_dir", how="inner")
           .select("source_dir", "npy_bytes", "ids_bytes"))
print("paired_sources =", df_pair.count())

schema = StructType([
    StructField("chunk_id", StringType(), False),
    StructField("embedding", ArrayType(FloatType()), False),
])

def decode_pairs(pdf_iter):
    # Decode .npy + ids inside executors (parallel)
    for pdf in pdf_iter:
        out = []
        for _, r in pdf.iterrows():
            ids = json.loads(bytes(r["ids_bytes"]).decode("utf-8"))
            vecs = np.load(io.BytesIO(bytes(r["npy_bytes"])))
            out.extend((cid, v.astype("float32").tolist()) for cid, v in zip(ids, vecs))
        yield pd.DataFrame(out, columns=["chunk_id", "embedding"])

df_emb = df_pair.mapInPandas(decode_pairs, schema=schema)

In [ ]:
# Persist the three optimized_* tables
df_emb.write.mode("overwrite").saveAsTable("tdis_dev_data_catalog.tdir.optimized_embeddings_dmretriever33m")
df_chunks.write.mode("overwrite").saveAsTable("tdis_dev_data_catalog.tdir.optimized_chunks_text")

df_final = df_chunks.join(df_emb, on="chunk_id", how="inner")
df_final.write.mode("overwrite").saveAsTable("tdis_dev_data_catalog.tdir.optimized_rag_chunks")

In [ ]:
# Verify
print("chunks_text:", spark.table("tdis_dev_data_catalog.tdir.optimized_chunks_text").count())
print("embeddings :", spark.table("tdis_dev_data_catalog.tdir.optimized_embeddings_dmretriever33m").count())
print("rag_chunks :", spark.table("tdis_dev_data_catalog.tdir.optimized_rag_chunks").count())

spark.table("tdis_dev_data_catalog.tdir.optimized_embeddings_dmretriever33m") \
    .select(count("*").alias("rows"), countDistinct("chunk_id").alias("unique_chunk_id")).show()

## 2. Vector Search index `optimized_rag_chunks_vs`

Create a **Databricks Vector Search** index on `optimized_rag_chunks` using the precomputed
`embedding` column (self-managed embeddings, primary key `chunk_id`). This is typically done
via the Vector Search UI or the `databricks-vectorsearch` SDK against the endpoint, e.g.:

```python
# from databricks.vector_search.client import VectorSearchClient
# vsc = VectorSearchClient()
# vsc.create_delta_sync_index(
#     endpoint_name="<your-endpoint>",
#     index_name="tdis_dev_data_catalog.tdir.optimized_rag_chunks_vs",
#     source_table_name="tdis_dev_data_catalog.tdir.optimized_rag_chunks",
#     primary_key="chunk_id",
#     embedding_dimension=384,
#     embedding_vector_column="embedding",
#     pipeline_type="TRIGGERED",
# )
```

The query side reads this index via `scripts/dbx_vector_search.py` (`DEFAULT_INDEX_FQN`).

## 3. Build BM25 inverted-index tables

Tokenizes `optimized_rag_chunks.text` and builds the four `optimized_kw_*` tables consumed
by `scripts/dbx_keyword_search_bm25.py`.

In [ ]:
from pyspark.sql import functions as F

CHUNKS = "tdis_dev_data_catalog.tdir.optimized_rag_chunks"
T_POST = "tdis_dev_data_catalog.tdir.optimized_kw_postings"
T_DST  = "tdis_dev_data_catalog.tdir.optimized_kw_doc_stats"
T_DF   = "tdis_dev_data_catalog.tdir.optimized_kw_df"
T_META = "tdis_dev_data_catalog.tdir.optimized_kw_meta"

chunks = spark.table(CHUNKS).select("chunk_id", "text")

# Tokenize (simple analyzer): lowercase, split on non-alnum, drop short/numeric tokens
terms = (chunks
  .select("chunk_id",
          F.explode(F.split(F.lower(F.regexp_replace(F.col("text"), r"[^a-z0-9]+", " ")), r"\s+")).alias("term"))
  .filter(F.length("term") >= 3)
  .filter(~F.col("term").rlike(r"^\d+$")))

# Postings: tf per (term, chunk_id)
postings = terms.groupBy("term", "chunk_id").agg(F.count("*").cast("int").alias("tf"))
postings.write.mode("overwrite").saveAsTable(T_POST)

# Doc stats: doc_len per chunk
doc_stats = postings.groupBy("chunk_id").agg(F.sum("tf").cast("int").alias("doc_len"))
doc_stats.write.mode("overwrite").saveAsTable(T_DST)

# DF: documents containing term
df_table = postings.groupBy("term").agg(F.count("*").cast("long").alias("df"))
df_table.write.mode("overwrite").saveAsTable(T_DF)

# Meta: N and avgdl
meta = doc_stats.agg(F.count("*").cast("long").alias("N"), F.avg("doc_len").cast("double").alias("avgdl"))
meta.write.mode("overwrite").saveAsTable(T_META)

In [ ]:
print("postings ", spark.table(T_POST).count())
print("doc_stats", spark.table(T_DST).count())
print("df       ", spark.table(T_DF).count())
print("meta     ", spark.table(T_META).first())